# Notebook 01: DRKG Data Exploration

**Project:** Explainable Knowledge Graph-Based Drug Repurposing for Acute Brain Injury: Extending XAIPath to DRKG  
**Author:** Maha Attique  
**Date:** July 2026  

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import csv
import sys
import dgl
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()

print("Done")

/projectnb/chenggrp/Maha/drkg_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Done


## 2. Load DRKG Triplets

The full graph contains 97,238 entities across 13 node types and 5,874,261 triplets across 107 relation types, integrated from six source databases: DrugBank, Hetionet, GNBR, STRING, IntAct, and DGIdb.

In [2]:
drkg_file = '../data/drkg.tsv'
df = pd.read_csv(drkg_file, sep='\t', header=None, names=['source', 'relation', 'target'])

print(f"Total triplets: {len(df):,}")
print(f"Unique relation types: {df['relation'].nunique()}")
# print(f"\nFirst 5 rows:")
# df.head()

Total triplets: 5,874,261
Unique relation types: 107


## 3. Node Types

DRKG entities follow the format `NodeType::ID`. Extract and count all node types to verify the graph loaded correctly.

In [3]:

all_entities = pd.concat([df['source'], df['target']]).unique()
node_types = pd.Series([e.split('::')[0] for e in all_entities])
type_counts = node_types.value_counts()

print(f"Total unique entities: {len(all_entities):,}")
print(f"\nEntity counts by node type:")
print(type_counts)

Total unique entities: 97,238

Entity counts by node type:
Gene                   39220
Compound               24313
Biological Process     11381
Side Effect             5701
Disease                 5103
Atc                     4048
Molecular Function      2884
Pathway                 1822
Cellular Component      1391
Symptom                  415
Anatomy                  400
Pharmacologic Class      345
Tax                      215
Name: count, dtype: int64


## 4. Compound-Disease Relations

For drug repurposing, the focus is on the edges connecting Compound nodes (drugs) to Disease nodes. Two relation types represent treatment relationships in DRKG:

- `Hetionet::CtD::Compound:Disease` — curated treatment relationships from Hetionet
- `GNBR::T::Compound:Disease` — treatment relationships extracted from biomedical literature (GNBR)

These are the same two relation types used in DRKG's original COVID-19 drug repurposing case study.

In [4]:

treatment_relations = [
    'Hetionet::CtD::Compound:Disease',
    'GNBR::T::Compound:Disease'
]

for rel in treatment_relations:
    count = (df['relation'] == rel).sum()
    print(f"{rel}: {count:,} triplets")

compound_disease = df[
    (df['source'].str.startswith('Compound::') & df['target'].str.startswith('Disease::')) |
    (df['source'].str.startswith('Disease::') & df['target'].str.startswith('Compound::'))
]
print(f"\nTotal Compound-Disease triplets (all relation types): {len(compound_disease):,}")
print(f"\nAll Compound-Disease relation types:")
print(compound_disease['relation'].value_counts())

Hetionet::CtD::Compound:Disease: 755 triplets
GNBR::T::Compound:Disease: 54,020 triplets

Total Compound-Disease triplets (all relation types): 83,895

All Compound-Disease relation types:
relation
GNBR::T::Compound:Disease             54020
GNBR::Sa::Compound:Disease            16923
DRUGBANK::treats::Compound:Disease     4968
GNBR::Pa::Compound:Disease             2619
GNBR::C::Compound:Disease              1739
GNBR::J::Compound:Disease              1020
GNBR::Pr::Compound:Disease              966
Hetionet::CtD::Compound:Disease         755
GNBR::Mp::Compound:Disease              495
Hetionet::CpD::Compound:Disease         390
Name: count, dtype: int64


## 5. Identify Target Disease Nodes for Acute Brain Injury

DRKG encodes standard diseases using MESH ontology IDs instead of plain text names (e.g., `Disease::MESH:D020521`). COVID-19-related entities are  added directly by name from a preprint dataset.

### How I identify target disease nodes
1. Examined `embed/entities.tsv`, which maps all DRKG entity names to their embedding indices
2. Looked up known MESH IDs for stroke, intracranial hemorrhage, and traumatic brain injury using the NLM MeSH Browser (https://meshb.nlm.nih.gov/)
3. Verified each MESH ID exists in `entities.tsv` before including it
4. Cross-referenced against DRKG triplets to confirm each node has edges in the graph

### Target diseases
Acute brain injury categories central to neurocritical care patient population at Boston Medical Center:
- **Ischemic stroke:** Stroke (D020521), Cerebral Infarction (D002544)
- **Intracranial hemorrhage:** Intracranial Hemorrhages (D020300), Intracranial Hemorrhage Hypertensive (D020520), Cerebral Hemorrhage (D002538)
- **Traumatic brain injury / general brain injury:** Brain Injuries (D001930)
- **Hemorrhage (general):** D006470 — included as a broader hemorrhagic disease node

In [5]:
brain_injury_disease_list = [
    'Disease::MESH:D020521',  # Stroke
    'Disease::MESH:D002544',  # Cerebral Infarction (ischemic stroke)
    'Disease::MESH:D020300',  # Intracranial Hemorrhages
    'Disease::MESH:D020520',  # Intracranial Hemorrhage, Hypertensive
    'Disease::MESH:D002538',  # Cerebral Hemorrhage
    'Disease::MESH:D001930',  # Brain Injuries
    'Disease::MESH:D006470',  # Hemorrhage
]

print(f"Target disease nodes: {len(brain_injury_disease_list)}")
for d in brain_injury_disease_list:
    print(f"  {d}")

Target disease nodes: 7
  Disease::MESH:D020521
  Disease::MESH:D002544
  Disease::MESH:D020300
  Disease::MESH:D020520
  Disease::MESH:D002538
  Disease::MESH:D001930
  Disease::MESH:D006470


## 6. Verifying Disease Nodes Exist in DRKG

In [6]:
print("Triplet counts per target disease node:\n")
for disease in brain_injury_disease_list:
    count = ((df['source'] == disease) | (df['target'] == disease)).sum()
    print(f"  {disease}: {count:,} triplets")

Triplet counts per target disease node:

  Disease::MESH:D020521: 442 triplets
  Disease::MESH:D002544: 273 triplets
  Disease::MESH:D020300: 21 triplets
  Disease::MESH:D020520: 11 triplets
  Disease::MESH:D002538: 7 triplets
  Disease::MESH:D001930: 290 triplets
  Disease::MESH:D006470: 396 triplets


## 7. Candidate Drug List

Following DRKG's original repurposing methodology, I use FDA-approved drugs from DrugBank as candidate compounds, excluding molecules with weight < 250 daltons. This list is provided in `drug_repurpose/infer_drug.tsv` and contains 8,104 candidates.

In [7]:
drug_list = []
with open("../drug_repurpose/infer_drug.tsv", newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile, delimiter='\t', fieldnames=['drug', 'ids'])
    for row_val in reader:
        drug_list.append(row_val['drug'])

print(f"Total candidate drugs: {len(drug_list):,}")

Total candidate drugs: 8,104


## 8. Pretrained TransE Embeddings Load Correctly

DRKG ships with pretrained TransE embeddings. Verify these load correctly because they will be Baseline 1 in my benchmarking pipeline.

In [8]:
entity_idmap_file = '../data/embed/entities.tsv'
relation_idmap_file = '../data/embed/relations.tsv'

entity_map = {}
entity_id_map = {}
relation_map = {}

with open(entity_idmap_file, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile, delimiter='\t', fieldnames=['name', 'id'])
    for row_val in reader:
        entity_map[row_val['name']] = int(row_val['id'])
        entity_id_map[int(row_val['id'])] = row_val['name']

with open(relation_idmap_file, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile, delimiter='\t', fieldnames=['name', 'id'])
    for row_val in reader:
        relation_map[row_val['name']] = int(row_val['id'])

print(f"Total entities in embedding map: {len(entity_map):,}")
print(f"Total relations in embedding map: {len(relation_map):,}")

# Load TransE embeddings
import torch as th
entity_emb = np.load('../data/embed/DRKG_TransE_l2_entity.npy')
rel_emb = np.load('../data/embed/DRKG_TransE_l2_relation.npy')

print(f"\nEntity embedding matrix shape: {entity_emb.shape}")
print(f"Relation embedding matrix shape: {rel_emb.shape}")

Total entities in embedding map: 97,238
Total relations in embedding map: 107

Entity embedding matrix shape: (97238, 400)
Relation embedding matrix shape: (107, 400)


## 9. Mapping Disease and Drug Nodes to Embedding IDs

In [9]:
print("Disease node embedding IDs:")
disease_ids = []
for disease in brain_injury_disease_list:
    if disease in entity_map:
        eid = entity_map[disease]
        disease_ids.append(eid)
        print(f"  {disease} -> embedding ID {eid}")
    else:
        print(f"  WARNING: {disease} not found in entity map")

# Map drug nodes to embedding IDs
drug_ids = []
missing_drugs = 0
for drug in drug_list:
    if drug in entity_map:
        drug_ids.append(entity_map[drug])
    else:
        missing_drugs += 1

print(f"\nCandidate drugs mapped: {len(drug_ids):,}")

Disease node embedding IDs:
  Disease::MESH:D020521 -> embedding ID 25242
  Disease::MESH:D002544 -> embedding ID 43694
  Disease::MESH:D020300 -> embedding ID 26221
  Disease::MESH:D020520 -> embedding ID 45320
  Disease::MESH:D002538 -> embedding ID 46499
  Disease::MESH:D001930 -> embedding ID 43732
  Disease::MESH:D006470 -> embedding ID 25108

Candidate drugs mapped: 8,104


## 10. Summary

Findings from the preliminary analysis:

- 97,238 entities, 5,874,261 triplets, 107 relation types
- 7 target disease nodes identified for acute brain injury (stroke, ICH, TBI)
- All target disease nodes verified present in graph triplets and embedding map
- 8,104 candidate drugs loaded from DrugBank FDA-approved list
- Pretrained TransE embeddings loaded successfully

### Next steps
- Score all candidate drug-disease pairs using pretrained TransE embeddings
- Train DistMult and ComplEx baselines via DGL-KE
- Train GraphSAGE link prediction model